In [2]:
import scanpy as sc

In [28]:
adata=sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/3sample_smalldataset_test_script_model_nolineage_preprocessed.h5ad")

In [17]:
lineage=sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/3sample_smalldataset_test_script_model.h5ad")

In [29]:
adata

AnnData object with n_obs × n_vars = 115549 × 29760
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'orig.ident', 'dataset', 'leiden_r1.0', 'is_original_query', 'original_query_id', 'lineage_pred', 'lineage_uncert', 'lineage', 'reanno_pred', 'reanno_uncert'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'leiden', 'leiden_r1.0_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap', 'X_umap_lineage', 'X_umap_reanno'
    varm: 'PCs'
    layers: 'counts', 'logcounts'
    obsp: 'connectivities', 'distances'

In [30]:
adata.obs['reanno_pred']

10X357_2:GCCCAGAGTGACCGTC    Oligodendrocyte
10X221_5:GGTGTTACACGCGCTA    Oligodendrocyte
10X319_1:CTGAGCGAGTATAGAC    Oligodendrocyte
10X387_2:TCGCACTAGTGCAAAT    Oligodendrocyte
10X393_7:TATCTGTAGCACTCTA    Oligodendrocyte
                                  ...       
SRX300884_SRX300884                       4C
SRX300876_SRX300876                   Oocyte
SRX300882_SRX300882                       2C
SRX300886_SRX300886                       4C
SRX300877_SRX300877                       2C
Name: reanno_pred, Length: 115549, dtype: category
Categories (163, object): ['2C', '4C', '8C', 'AVE', ..., 'Vascular', 'YS.endoderm', 'YS.mesoderm_1', 'YS.mesoderm_2']

In [31]:
adata.obs['reanno']=lineage.obs['reanno']

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd

# 创建交叉表
cross_tab = pd.crosstab(adata.obs['lineage_pred'], 
                        adata.obs['lineage'])

print("=" * 80)
print("簇间一致性分析（共同细胞数 / lineage细胞数）")
print("=" * 80)

# 分析每个簇的对应关系
for cluster in cross_tab.index:
    if cluster in cross_tab.columns:
        # 共同细胞数：在两个注释中都标记为该簇的细胞
        common_cells = cross_tab.loc[cluster, cluster]
        # lineage中该簇的总细胞数
        lineage_total = cross_tab[cluster].sum()
        # 比例
        proportion = common_cells / lineage_total * 100 if lineage_total > 0 else 0
        
        print(f"{cluster}: {common_cells}/{lineage_total} cells ({proportion:.1f}%)")
    else:
        # 如果该簇在lineage中不存在
        lineage_total = 0
        print(f"{cluster}: 0/0 cells (0.0%) - 在lineage中不存在")

print("\n" + "=" * 80)
print("详细统计信息")
print("=" * 80)

# 额外统计：lineage中有但另一个注释中没有的簇
lineage_only_clusters = set(cross_tab.columns) - set(cross_tab.index)
if lineage_only_clusters:
    print(f"\n只在lineage中存在的簇:")
    for cluster in lineage_only_clusters:
        lineage_total = cross_tab[cluster].sum()
        print(f"  {cluster}: {lineage_total} 个细胞")

# 统计：另一个注释中有但lineage中没有的簇
other_only_clusters = set(cross_tab.index) - set(cross_tab.columns)
if other_only_clusters:
    print(f"\n只在lineage_pred中存在的簇:")
    for cluster in other_only_clusters:
        other_total = cross_tab.loc[cluster].sum()
        print(f"  {cluster}: {other_total} 个细胞")

# 总体统计
total_common_cells = sum(cross_tab.loc[cluster, cluster] for cluster in cross_tab.index if cluster in cross_tab.columns)
total_lineage_cells = cross_tab.sum().sum()
overall_proportion = total_common_cells / total_lineage_cells * 100

print(f"\n总体统计:")
print(f"总共同细胞数: {total_common_cells}")
print(f"lineage总细胞数: {total_lineage_cells}")
print(f"总体一致性: {overall_proportion:.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import pandas as pd
import re

def simplify_annotation_advanced(annotation):
    """
    增强版的注释简化函数，支持多种归类规则
    """
    if pd.isna(annotation):
        return annotation
    
    annotation_str = str(annotation)
    
    # 第一步：统一分隔符，将空格和点号都统一为点号
    annotation_str = re.sub(r'[\s\.]+', '.', annotation_str)
    
    # 第二步：移除VGLUT1, VGLUT2等标记
    annotation_str = re.sub(r'\.VGLUT\d+', '', annotation_str)
    annotation_str = re.sub(r'\s+VGLUT\d+', '', annotation_str)
    
    # 第三步：移除其他类似的标记（根据需要添加）
    annotation_str = re.sub(r'\.\w+\d+\s*$', '', annotation_str)
    
    # 第四步：按点分割，保留主要结构
    parts = annotation_str.split('.')
    # 过滤空字符串
    parts = [part for part in parts if part]
    
    if len(parts) >= 3:
        # 保留前三个主要部分
        simplified = '.'.join(parts[:3])
        # 清理可能的空格
        simplified = simplified.strip()
        return simplified
    elif len(parts) > 0:
        return '.'.join(parts)
    else:
        return annotation_str.strip()

def normalize_separators(annotation):
    """
    专门用于统一分隔符的函数
    """
    if pd.isna(annotation):
        return annotation
    
    annotation_str = str(annotation)
    # 将各种分隔符（空格、点号、逗号等）统一为点号
    annotation_str = re.sub(r'[\s\.,;]+', '.', annotation_str)
    # 移除首尾的点号
    annotation_str = annotation_str.strip('.')
    return annotation_str

# 使用增强版简化函数
adata.obs['reanno_simplified'] = adata.obs['reanno'].apply(simplify_annotation_advanced)
adata.obs['reanno_pred_simplified'] = adata.obs['reanno_pred'].apply(simplify_annotation_advanced)

# 创建交叉表（使用简化后的注释）
cross_tab = pd.crosstab(adata.obs['reanno_pred_simplified'], 
                        adata.obs['reanno_simplified'])

print("=" * 80)
print("簇间一致性分析（共同细胞数 / lineage细胞数）- 简化注释版本")
print("=" * 80)

# 分析每个簇的对应关系
for cluster in cross_tab.index:
    if cluster in cross_tab.columns:
        # 共同细胞数：在两个注释中都标记为该簇的细胞
        common_cells = cross_tab.loc[cluster, cluster]
        # lineage中该簇的总细胞数
        lineage_total = cross_tab[cluster].sum()
        # 比例
        proportion = common_cells / lineage_total * 100 if lineage_total > 0 else 0
        
        print(f"{cluster}: {common_cells}/{lineage_total} cells ({proportion:.1f}%)")
    else:
        # 如果该簇在lineage中不存在
        lineage_total = 0
        print(f"{cluster}: 0/0 cells (0.0%) - 在lineage中不存在")

print("\n" + "=" * 80)
print("详细统计信息")
print("=" * 80)

# 额外统计：lineage中有但另一个注释中没有的簇
lineage_only_clusters = set(cross_tab.columns) - set(cross_tab.index)
if lineage_only_clusters:
    print(f"\n只在lineage中存在的簇:")
    for cluster in lineage_only_clusters:
        lineage_total = cross_tab[cluster].sum()
        print(f"  {cluster}: {lineage_total} 个细胞")

# 统计：另一个注释中有但lineage中没有的簇
other_only_clusters = set(cross_tab.index) - set(cross_tab.columns)
if other_only_clusters:
    print(f"\n只在lineage_pred中存在的簇:")
    for cluster in other_only_clusters:
        other_total = cross_tab.loc[cluster].sum()
        print(f"  {cluster}: {other_total} 个细胞")

# 总体统计
total_common_cells = sum(cross_tab.loc[cluster, cluster] for cluster in cross_tab.index if cluster in cross_tab.columns)
total_lineage_cells = cross_tab.sum().sum()
overall_proportion = total_common_cells / total_lineage_cells * 100

print(f"\n总体统计:")
print(f"总共同细胞数: {total_common_cells}")
print(f"lineage总细胞数: {total_lineage_cells}")
print(f"总体一致性: {overall_proportion:.1f}%")

# 可选：显示原始注释和简化注释的映射关系
print("\n" + "=" * 80)
print("注释简化映射关系")
print("=" * 80)

original_to_simplified = adata.obs[['reanno', 'reanno_simplified']].drop_duplicates()
print("reanno 原始注释 -> 简化注释:")
for _, row in original_to_simplified.iterrows():
    print(f"  {row['reanno']} -> {row['reanno_simplified']}")

original_to_simplified_pred = adata.obs[['reanno_pred', 'reanno_pred_simplified']].drop_duplicates()
print("\nreanno_pred 原始注释 -> 简化注释:")
for _, row in original_to_simplified_pred.iterrows():
    print(f"  {row['reanno_pred']} -> {row['reanno_pred_simplified']}")

# 新增：显示统一分隔符后的效果
print("\n" + "=" * 80)
print("分隔符统一测试示例")
print("=" * 80)
test_cases = ["Upper-layer intratelencephalic Neuron", "Upper-layer.intratelencephalic.Neuron"]
for test in test_cases:
    normalized = normalize_separators(test)
    simplified = simplify_annotation_advanced(test)
    print(f"原始: '{test}' -> 统一分隔符: '{normalized}' -> 简化: '{simplified}'")